# IKG Column Lineage Explorer

Interactive **Sigma.js** graph that traces any column from target → source
across all IKG tables.

**Prerequisites:**  
- Run `ikg_column_lineage_master_auto_refresh.ipynb` first to generate
  the lineage table in Greenplum (or export a CSV from it).

**This notebook does two things:**  
1. Reads the lineage data (from Greenplum or a CSV file)  
2. Writes a self-contained `ikg_lineage_explorer.html` you can open in any browser


## 1. Imports


In [ ]:
import os, json
import pandas as pd
from pathlib import Path
from IPython.display import IFrame, display as ipy_display


## 2. Configuration

Choose **one** data source — Greenplum table or a local CSV/Excel file.


In [ ]:
# ── Data source ──────────────────────────────────────────────────────────
# Set USE_GREENPLUM = True to query the live table, or False to load a file.

USE_GREENPLUM = False

# If USE_GREENPLUM = False, point to the CSV or Excel exported from the main notebook
LOCAL_FILE = 'ikg_lineage_master.csv'    # or .xlsx

# Greenplum connection (only needed when USE_GREENPLUM = True)
GP_HOST     = 'greenplum-rdsp.zur.swissbank.com'
GP_PORT     = 5432
GP_DB       = 'gpadmin'
GP_USER     = 'gpadmin'
GP_SCHEMA   = 'sandbox_prj_smart_insights'
GP_TABLE    = 'ikg_column_lineage_master_auto_refresh'
GP_PASSWORD = ''   # set here or enter below when prompted

# Output HTML file
HTML_OUT = 'ikg_lineage_explorer.html'


## 3. Load Lineage Data


In [ ]:
if USE_GREENPLUM:
    import getpass, sqlalchemy
    if not GP_PASSWORD:
        GP_PASSWORD = getpass.getpass('Greenplum password: ')
    engine = sqlalchemy.create_engine(
        f'postgresql+psycopg2://{GP_USER}:{GP_PASSWORD}@{GP_HOST}:{GP_PORT}/{GP_DB}'
    )
    df = pd.read_sql(f'SELECT * FROM {GP_SCHEMA}.{GP_TABLE}', engine)
    print(f'Loaded {len(df):,} rows from Greenplum: {GP_SCHEMA}.{GP_TABLE}')
else:
    p = Path(LOCAL_FILE)
    if not p.exists():
        raise FileNotFoundError(
            f'File not found: {LOCAL_FILE}\n'
            'Run ikg_column_lineage_master_auto_refresh.ipynb first, '
            'then export the lineage DataFrame:\n'
            '  df.to_csv("ikg_lineage_master.csv", index=False)'
        )
    if p.suffix == '.csv':
        df = pd.read_csv(p, dtype=str).fillna('')
    else:
        df = pd.read_excel(p, dtype=str).fillna('')
    print(f'Loaded {len(df):,} rows from {LOCAL_FILE}')

df.head(3)


## 4. Export CSV for the HTML Explorer

The HTML file loads data from a CSV that you point it to via the **📂 Load CSV** button.
This cell writes that CSV next to the HTML file.


In [ ]:
csv_out = Path(HTML_OUT).stem + '_data.csv'
df.to_csv(csv_out, index=False)
print(f'CSV written: {csv_out}  ({len(df):,} rows)')
print(f'You will load this file in the HTML explorer.')


## 5. Write HTML Lineage Explorer

Writes a self-contained `ikg_lineage_explorer.html`.  
Open it in Chrome / Edge / Firefox — no server required.


In [ ]:
HTML_CONTENT = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>IKG Column Lineage Explorer</title>
<script src="https://cdnjs.cloudflare.com/ajax/libs/sigma.js/2.4.0/sigma.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/graphology/0.25.4/graphology.umd.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/xlsx/0.18.5/xlsx.full.min.js"></script>
<style>
* { box-sizing: border-box; margin: 0; padding: 0; }
body { font-family: 'Segoe UI', Arial, sans-serif; height: 100vh; display: flex; flex-direction: column; overflow: hidden; background: #f0f4f8; }

/* TOP BAR */
#topbar {
  background: linear-gradient(135deg, #1a1a2e 0%, #16213e 60%, #0f3460 100%);
  color: #fff; padding: 10px 20px; display: flex; align-items: center; gap: 14px;
  box-shadow: 0 2px 10px rgba(0,0,0,.4); flex-shrink: 0; z-index: 20;
}
#topbar h1 { font-size: 17px; font-weight: 700; letter-spacing: .4px; white-space: nowrap; }
#topbar h1 em { color: #e94560; font-style: normal; }
.vsep { width: 1px; height: 28px; background: rgba(255,255,255,.2); flex-shrink: 0; }
.file-btn {
  cursor: pointer; background: rgba(255,255,255,.12); padding: 6px 14px;
  border-radius: 6px; font-size: 13px; border: 1px solid rgba(255,255,255,.2);
  color: #fff; white-space: nowrap; transition: background .15s;
}
.file-btn:hover { background: rgba(255,255,255,.22); }
#status { font-size: 12px; color: rgba(255,255,255,.55); white-space: nowrap; margin-left: auto; }

/* MAIN ROW */
#main { display: flex; flex: 1; overflow: hidden; }

/* LEFT PANEL */
#left {
  width: 270px; background: #fff; border-right: 1px solid #dde3ec;
  display: flex; flex-direction: column; flex-shrink: 0; overflow: hidden;
}
#left-hdr {
  background: #1a1a2e; color: #fff; padding: 10px 14px;
  font-size: 13px; font-weight: 600; flex-shrink: 0;
}
.step-blk { padding: 11px 14px; border-bottom: 1px solid #eef0f5; flex-shrink: 0; }
.step-lbl {
  font-size: 11px; font-weight: 700; text-transform: uppercase;
  letter-spacing: .5px; color: #8a95a8; margin-bottom: 7px;
  display: flex; align-items: center; gap: 6px;
}
.snum {
  width: 18px; height: 18px; border-radius: 50%; font-size: 10px; font-weight: 800;
  display: flex; align-items: center; justify-content: center; flex-shrink: 0;
  color: #fff; background: #c5cad4;
}
.snum.active { background: #e94560; }
.snum.done   { background: #20c997; }

select.dsel {
  width: 100%; padding: 7px 10px; border: 1.5px solid #dde3ec; border-radius: 6px;
  font-size: 13px; color: #222; outline: none; background: #f8fafc;
  cursor: pointer; transition: border-color .15s;
}
select.dsel:focus { border-color: #e94560; }
select.dsel:disabled { color: #b0b8c4; cursor: default; }

#tbl-scroll { flex: 1; overflow-y: auto; }
.tbl-item {
  padding: 9px 14px; font-size: 13px; cursor: pointer;
  border-bottom: 1px solid #f0f2f6; color: #2d3a4a;
  transition: background .1s; display: flex; align-items: center; gap: 8px;
}
.tbl-item:hover  { background: #eef2ff; }
.tbl-item.active { background: #e2e9ff; font-weight: 600; color: #1a1a2e; }
.tbl-note { color: #aaa; font-size: 13px; padding: 16px 14px; text-align: center; line-height: 1.5; }

#btn-trace {
  margin: 12px 14px 14px; padding: 9px; background: #e94560; color: #fff;
  border: none; border-radius: 7px; font-size: 14px; font-weight: 700;
  cursor: pointer; transition: background .15s; letter-spacing: .3px;
  box-shadow: 0 3px 8px rgba(233,69,96,.35); flex-shrink: 0;
}
#btn-trace:hover    { background: #c73652; }
#btn-trace:disabled { background: #c5cad4; cursor: default; box-shadow: none; }

/* GRAPH */
#gwrap { flex: 1; position: relative; background: #06061a; overflow: hidden; }
#sc    { width: 100%; height: 100%; }
#ghint {
  position: absolute; inset: 0; display: flex; flex-direction: column;
  align-items: center; justify-content: center; gap: 10px;
  color: rgba(255,255,255,.22); pointer-events: none; text-align: center;
}
#ghint .gi { font-size: 52px; }
#ghint p   { font-size: 14px; line-height: 1.6; }

#cam-ctrl { position: absolute; top: 12px; right: 12px; display: flex; gap: 6px; }
.cbtn {
  width: 32px; height: 32px; background: rgba(255,255,255,.12);
  border: 1px solid rgba(255,255,255,.2); color: #fff; border-radius: 6px;
  cursor: pointer; font-size: 17px; display: flex; align-items: center; justify-content: center;
}
.cbtn:hover { background: rgba(255,255,255,.24); }

#legend {
  position: absolute; bottom: 14px; left: 14px;
  background: rgba(12,14,38,.88); border-radius: 8px;
  padding: 10px 14px; display: none; backdrop-filter: blur(4px);
}
#legend h4 { color: rgba(255,255,255,.45); font-size: 10px; text-transform: uppercase; letter-spacing: .5px; margin-bottom: 7px; }
.lr { display: flex; align-items: center; gap: 8px; margin-bottom: 4px; font-size: 12px; color: rgba(255,255,255,.75); }
.ld { width: 11px; height: 11px; border-radius: 50%; flex-shrink: 0; }

#tip {
  position: absolute; background: rgba(12,14,38,.92); color: #fff;
  padding: 5px 10px; border-radius: 5px; font-size: 12px;
  display: none; pointer-events: none; white-space: nowrap; z-index: 30;
}
#spin-wrap {
  position: absolute; inset: 0; background: rgba(6,6,26,.7);
  display: none; align-items: center; justify-content: center; z-index: 50;
}
#spin-wrap.on { display: flex; }
.spin { width: 42px; height: 42px; border: 3px solid rgba(255,255,255,.15); border-top-color: #e94560; border-radius: 50%; animation: rot .7s linear infinite; }
@keyframes rot { to { transform: rotate(360deg); } }

/* RIGHT PANEL */
#rpanel {
  width: 330px; background: #fff; border-left: 1px solid #dde3ec;
  display: flex; flex-direction: column; flex-shrink: 0; overflow: hidden;
}
#rphdr { background: #1a1a2e; color: #fff; padding: 10px 14px; font-size: 13px; font-weight: 600; flex-shrink: 0; }
#rpbody { flex: 1; overflow-y: auto; padding: 12px; }
.rp-hint { color: #aaa; font-size: 13px; text-align: center; padding: 40px 14px; line-height: 1.6; }
.rp-hint span { font-size: 34px; display: block; margin-bottom: 10px; }

.dc { border: 1px solid #e8ecf2; border-radius: 7px; overflow: hidden; margin-bottom: 10px; }
.dch {
  background: #f5f7fb; padding: 7px 11px; font-size: 11px; font-weight: 700;
  text-transform: uppercase; letter-spacing: .4px; color: #5a6474;
  border-bottom: 1px solid #e8ecf2; display: flex; align-items: center; gap: 7px;
}
.dr { display: flex; padding: 5px 11px; border-bottom: 1px solid #f2f4f8; font-size: 13px; }
.dr:last-child { border-bottom: none; }
.dl { color: #8a95a8; width: 120px; flex-shrink: 0; font-size: 12px; }
.dv { color: #1a2232; word-break: break-word; font-weight: 500; }
.dv.empty { color: #c5cad4; font-style: italic; font-weight: 400; }
.dv.mono  { font-family: 'Courier New', monospace; font-size: 11px; background: #f5f7fb; padding: 2px 5px; border-radius: 3px; }

.badge { display: inline-block; padding: 2px 8px; border-radius: 10px; font-size: 11px; font-weight: 700; }
.bs { background:#e8f5e9;color:#2e7d32; }
.bj { background:#e3f2fd;color:#1565c0; }
.bw { background:#fff3e0;color:#e65100; }
.bh { background:#fce4ec;color:#c62828; }
.bv { background:#f3e5f5;color:#6a1b9a; }
.bx { background:#e0f7fa;color:#00695c; }

#tbl-scroll::-webkit-scrollbar,#rpbody::-webkit-scrollbar { width: 5px; }
#tbl-scroll::-webkit-scrollbar-thumb,#rpbody::-webkit-scrollbar-thumb { background:#ccd0d9;border-radius:3px; }
</style>
</head>
<body>

<div id="topbar">
  <h1>IKG Lineage <em>Explorer</em></h1>
  <div class="vsep"></div>
  <label class="file-btn">
    📂 Load CSV / Excel
    <input id="file-inp" type="file" accept=".csv,.xlsx,.xls" style="display:none" onchange="loadFile(this)">
  </label>
  <span id="status">Load the lineage CSV or Excel file to begin</span>
</div>

<div id="main">

  <!-- LEFT: column + table selection -->
  <div id="left">
    <div id="left-hdr">Select Column &amp; Table</div>

    <div class="step-blk">
      <div class="step-lbl"><span class="snum" id="sn1">1</span>Target Column</div>
      <select class="dsel" id="col-sel" disabled onchange="onColChange()">
        <option value="">— load file first —</option>
      </select>
    </div>

    <div class="step-blk" style="padding-bottom:8px">
      <div class="step-lbl"><span class="snum" id="sn2">2</span>Profile Table</div>
    </div>
    <div id="tbl-scroll">
      <div class="tbl-note">Choose a column first</div>
    </div>

    <button id="btn-trace" disabled onclick="runTrace()">▶&nbsp; Trace Lineage</button>
  </div>

  <!-- CENTRE: Sigma graph -->
  <div id="gwrap">
    <div id="sc"></div>
    <div id="ghint">
      <div class="gi">🔍</div>
      <p>Load a lineage file, pick a column<br>and a table, then click Trace.</p>
    </div>
    <div id="cam-ctrl">
      <button class="cbtn" title="Zoom in"  onclick="camZ(1.35)">+</button>
      <button class="cbtn" title="Zoom out" onclick="camZ(0.74)">−</button>
      <button class="cbtn" title="Fit view" onclick="camFit()">⊡</button>
    </div>
    <div id="legend"><h4>Schema</h4><div id="leg-inner"></div></div>
    <div id="tip"></div>
    <div id="spin-wrap"><div class="spin"></div></div>
  </div>

  <!-- RIGHT: detail panel -->
  <div id="rpanel">
    <div id="rphdr">Node Details</div>
    <div id="rpbody">
      <div class="rp-hint"><span>💡</span>Click any node to see its lineage details here.</div>
    </div>
  </div>

</div>

<script>
// ═══════════════════════════════════════════════════════════════════════════
//  STATE
// ═══════════════════════════════════════════════════════════════════════════
let ROWS = [];
let SEL_COL = '', SEL_TBL = '';
let sigInst = null, G = null;

// ═══════════════════════════════════════════════════════════════════════════
//  SCHEMA COLOURS
// ═══════════════════════════════════════════════════════════════════════════
const PALETTE = [
  ['core_ikg',                       '#e94560'],
  ['ikg_schema',                     '#e94560'],
  ['sandbox_prj_smart_insights',     '#e94560'],
  ['core_wma_shared',                '#0f9b8e'],
  ['edw_view_input_schema',          '#0f9b8e'],
  ['edw_input_schema',               '#4dabf7'],
  ['core_model',                     '#f5a623'],
  ['model_schema',                   '#f5a623'],
  ['core_nlg',                       '#7b68ee'],
  ['nlg_schema',                     '#7b68ee'],
  ['sandbox_prj_smart_relationship', '#20c997'],
  ['ikg_wealthx_schema',             '#20c997'],
];
function sColor(s) {
  const sl = (s||'').toLowerCase().trim();
  for (const [k,v] of PALETTE) if (sl===k||sl.includes(k)||k.includes(sl)) return v;
  return '#74b9ff';
}

// ═══════════════════════════════════════════════════════════════════════════
//  FILE LOADING
// ═══════════════════════════════════════════════════════════════════════════
function loadFile(inp) {
  const f = inp.files[0]; if (!f) return;
  setSt('Loading…');
  const rd = new FileReader();
  if (f.name.match(/\\.csv$/i)) {
    rd.onload = e => { ROWS = parseCSV(e.target.result); afterLoad(f.name); };
    rd.readAsText(f);
  } else {
    rd.onload = e => {
      try {
        const wb = XLSX.read(new Uint8Array(e.target.result),{type:'array'});
        const ws = wb.Sheets[wb.SheetNames[0]];
        ROWS = XLSX.utils.sheet_to_json(ws,{defval:''})
          .map(r=>{ const o={}; for(const k in r) o[k.trim()]=String(r[k]??'').trim(); return o; });
        afterLoad(f.name);
      } catch(ex) { setSt('xlsx error: '+ex.message); }
    };
    rd.readAsArrayBuffer(f);
  }
}

function afterLoad(name) {
  setSt(`✓ ${ROWS.length.toLocaleString()} rows — ${name}`);
  buildColDrop();
}
function setSt(m) { document.getElementById('status').textContent = m; }

// ═══════════════════════════════════════════════════════════════════════════
//  STEP 1 — column dropdown
//  Profile rows: target_table = sub_target_table AND target_table LIKE %_profile_curr_ikg
// ═══════════════════════════════════════════════════════════════════════════
function isProfile(r) {
  const tt  = (r.target_table     || '').toLowerCase().trim();
  const stt = (r.sub_target_table || '').toLowerCase().trim();
  return tt === stt && tt.endsWith('_profile_curr_ikg');
}

function buildColDrop() {
  const cols = [...new Set(
    ROWS.filter(isProfile).map(r=>r.target_column||'').filter(Boolean)
  )].sort();

  const el = document.getElementById('col-sel');
  el.innerHTML = `<option value="">— select column (${cols.length}) —</option>`
    + cols.map(c=>`<option value="${e$(c)}">${e$(c)}</option>`).join('');
  el.disabled = false;
  document.getElementById('sn1').className = 'snum active';
}

// ═══════════════════════════════════════════════════════════════════════════
//  STEP 2 — table list
// ═══════════════════════════════════════════════════════════════════════════
function onColChange() {
  SEL_COL = document.getElementById('col-sel').value;
  SEL_TBL = '';
  document.getElementById('btn-trace').disabled = true;
  document.getElementById('sn2').className = 'snum';

  const box = document.getElementById('tbl-scroll');
  if (!SEL_COL) { box.innerHTML = '<div class="tbl-note">Choose a column first</div>'; return; }

  const tbls = [...new Set(
    ROWS.filter(r => isProfile(r) && (r.target_column||'')=== SEL_COL)
        .map(r=>r.target_table||'').filter(Boolean)
  )].sort();

  if (!tbls.length) { box.innerHTML = '<div class="tbl-note">No profile tables found</div>'; return; }

  document.getElementById('sn2').className = 'snum active';
  box.innerHTML = tbls.map(t =>
    `<div class="tbl-item" onclick="selTbl(this,'${e$(t)}')">📋&nbsp;${e$(t)}</div>`
  ).join('');
}

function selTbl(el, tbl) {
  document.querySelectorAll('.tbl-item').forEach(x=>x.classList.remove('active'));
  el.classList.add('active');
  SEL_TBL = tbl;
  document.getElementById('btn-trace').disabled = false;
}

// ═══════════════════════════════════════════════════════════════════════════
//  STEP 3 — BFS lineage traversal
// ═══════════════════════════════════════════════════════════════════════════
function runTrace() {
  if (!SEL_COL || !SEL_TBL) return;
  document.getElementById('spin-wrap').classList.add('on');
  document.getElementById('ghint').style.display = 'none';
  setTimeout(() => {
    try {
      const data = traceLineage(SEL_COL, SEL_TBL);
      drawGraph(data);
    } catch(ex) { setSt('Error: '+ex.message); console.error(ex); }
    document.getElementById('spin-wrap').classList.remove('on');
  }, 30);
}

// Lookup rows where sub_target_table=tbl AND target_column=col
function getRows(tbl, col) {
  const t=tbl.toLowerCase(), c=col.toLowerCase();
  return ROWS.filter(r=>
    (r.sub_target_table||'').toLowerCase()===t &&
    (r.target_column   ||'').toLowerCase()===c
  );
}

// Does any row have sub_target_table=tbl (for any col)?
function hasSTT(tbl) {
  const t=tbl.toLowerCase();
  return ROWS.some(r=>(r.sub_target_table||'').toLowerCase()===t);
}

function resolveSchema(tbl) {
  const t = tbl.toLowerCase();
  for (const r of ROWS) {
    if ((r.source_table||'').toLowerCase()===t && r.source_schema) return r.source_schema;
    if ((r.sub_target_table||'').toLowerCase()===t && r.sub_target_schema) return r.sub_target_schema;
  }
  return '';
}

function traceLineage(startCol, startTbl) {
  const nid = (tbl,col) => `${(tbl||'?').toLowerCase()}::${(col||'?').toLowerCase()}`;

  const nodes    = new Map();   // id -> {tbl,col,schema,isStart}
  const edgesMap = new Map();   // "from->to" -> {from,to,rows:[]}
  const nRows    = new Map();   // id -> rows[]
  const visited  = new Set();
  const queue    = [];

  function addNode(tbl,col,schema,isStart) {
    const id = nid(tbl,col);
    if (!nodes.has(id)) { nodes.set(id,{tbl,col,schema:schema||'',isStart:!!isStart}); nRows.set(id,[]); }
    return id;
  }
  function addEdge(from,to,rows) {
    const k=`${from}→${to}`;
    if (!edgesMap.has(k)) edgesMap.set(k,{from,to,rows:[]});
    edgesMap.get(k).rows.push(...rows);
  }

  // Seed from the chosen profile table + column
  const seedRows = ROWS.filter(r=>
    isProfile(r) &&
    (r.target_column||'').toLowerCase() === startCol.toLowerCase() &&
    (r.target_table ||'').toLowerCase() === startTbl.toLowerCase()
  );
  const startId = addNode(startTbl, startCol, seedRows[0]?.target_schema||'', true);
  nRows.get(startId).push(...seedRows);
  visited.add(startId);

  seedRows.forEach(r => {
    queue.push({ srcTbl: r.source_table||'', srcCol: r.source_column||'', parentId: startId, triggerRows:[r] });
  });

  let iter=0;
  while (queue.length && iter++<800) {
    const { srcTbl, srcCol, parentId, triggerRows } = queue.shift();

    // ── blank source_table: look up srcCol as target_column ───────────────
    if (!srcTbl && srcCol) {
      const upRows = ROWS.filter(r=>
        isProfile(r) &&
        (r.target_column||'').toLowerCase()===srcCol.toLowerCase()
      );
      upRows.forEach(r=>{
        const uid = addNode(r.target_table||'', srcCol, r.target_schema||'', false);
        addEdge(uid, parentId, triggerRows);
        nRows.get(uid).push(r);
        if (!visited.has(uid)) {
          visited.add(uid);
          const sub = getRows(r.target_table||'', srcCol);
          nRows.get(uid).push(...sub);
          sub.forEach(sr=>queue.push({srcTbl:sr.source_table||'',srcCol:sr.source_column||'',parentId:uid,triggerRows:[sr]}));
        }
      });
      continue;
    }
    if (!srcTbl && !srcCol) continue;

    const schema = resolveSchema(srcTbl);
    const srcId  = addNode(srcTbl, srcCol, schema, false);
    addEdge(srcId, parentId, triggerRows);

    if (visited.has(srcId)) continue;
    visited.add(srcId);

    const sub = getRows(srcTbl, srcCol);
    nRows.get(srcId).push(...sub);

    if (!sub.length) {
      // Base source — check if srcCol itself has lineage under a blank source_table
      const blankSrcRows = ROWS.filter(r=>
        (r.sub_target_table||'').toLowerCase()===srcTbl.toLowerCase() &&
        (r.target_column   ||'').toLowerCase()===srcCol.toLowerCase() &&
        !(r.source_table||'').trim()
      );
      blankSrcRows.forEach(r=>{
        if ((r.source_column||'').trim())
          queue.push({srcTbl:'',srcCol:r.source_column,parentId:srcId,triggerRows:[r]});
      });
      continue;
    }

    sub.forEach(sr=>
      queue.push({srcTbl:sr.source_table||'',srcCol:sr.source_column||'',parentId:srcId,triggerRows:[sr]})
    );
  }

  return {
    nodes: [...nodes.entries()].map(([id,d])=>({id,...d})),
    edges: [...edgesMap.values()],
    nRows
  };
}

// ═══════════════════════════════════════════════════════════════════════════
//  DRAW GRAPH
// ═══════════════════════════════════════════════════════════════════════════
function drawGraph({ nodes, edges, nRows }) {
  if (sigInst) { sigInst.kill(); sigInst=null; }
  document.getElementById('legend').style.display='none';

  if (!nodes.length) { setSt('No lineage found.'); document.getElementById('ghint').style.display=''; return; }

  G = new graphology.Graph({ type:'directed', multi:false });

  // ── Topological level assignment (Kahn's) ────────────────────────────────
  const adjOut = new Map(), indeg = new Map();
  nodes.forEach(n=>{ adjOut.set(n.id,[]); indeg.set(n.id,0); });
  edges.forEach(e=>{
    if (adjOut.has(e.from)&&adjOut.has(e.to)) {
      adjOut.get(e.from).push(e.to);
      indeg.set(e.to,(indeg.get(e.to)||0)+1);
    }
  });
  const levels=new Map(), q=[];
  indeg.forEach((d,id)=>{ if(d===0) q.push(id); });
  while(q.length) {
    const id=q.shift(), lv=levels.get(id)||0;
    (adjOut.get(id)||[]).forEach(t=>{
      levels.set(t,Math.max(levels.get(t)||0,lv+1));
      indeg.set(t,indeg.get(t)-1);
      if(indeg.get(t)===0) q.push(t);
    });
  }
  nodes.forEach(n=>{ if(!levels.has(n.id)) levels.set(n.id,0); });

  const byLv=new Map();
  nodes.forEach(n=>{ const lv=levels.get(n.id)||0; if(!byLv.has(lv)) byLv.set(lv,[]); byLv.get(lv).push(n); });
  const maxLv=Math.max(...byLv.keys());

  // ── Position nodes ────────────────────────────────────────────────────────
  const W=900,H=620;
  const xStep=maxLv>0?W/maxLv:W/2;
  const schemas=new Map();

  byLv.forEach((ns,lv)=>{
    const x=(maxLv-lv)*xStep;   // flip: target on right, sources on left
    ns.forEach((n,i)=>{
      const y=(i+1)*(H/(ns.length+1));
      const color=sColor(n.schema);
      const sk=(n.schema||'unknown').toLowerCase();
      if(!schemas.has(sk)) schemas.set(sk,{label:n.schema||'unknown',color});
      G.addNode(n.id,{
        x,y,size:n.isStart?18:11,color,
        label:`${n.tbl}\\n${n.col}`,
        _tbl:n.tbl,_col:n.col,_schema:n.schema
      });
    });
  });

  // ── Add edges ─────────────────────────────────────────────────────────────
  const eSeen=new Set();
  edges.forEach(e=>{
    const k=`${e.from}→${e.to}`;
    if(G.hasNode(e.from)&&G.hasNode(e.to)&&!eSeen.has(k)){
      eSeen.add(k);
      G.addEdge(e.from,e.to,{size:2,color:'rgba(140,170,255,0.38)',type:'arrow'});
    }
  });

  // ── Sigma ─────────────────────────────────────────────────────────────────
  const container=document.getElementById('sc');
  sigInst=new Sigma(G,container,{
    renderEdgeLabels:false,defaultEdgeType:'arrow',
    labelFont:'Segoe UI,Arial',labelSize:11,
    labelColor:{color:'#fff'},
    minCameraRatio:0.04,maxCameraRatio:12,
  });

  // Click → detail panel
  sigInst.on('clickNode',({node})=>showDetail(node,nRows));

  // Hover tooltip
  const tip=document.getElementById('tip');
  sigInst.on('enterNode',({node,event})=>{
    const a=G.getNodeAttributes(node);
    tip.textContent=`${a._tbl} · ${a._col}`;
    tip.style.display='block';
    moveTip(event.original);
  });
  sigInst.on('leaveNode',()=>{ tip.style.display='none'; });
  container.addEventListener('mousemove',ev=>{ if(tip.style.display!=='none') moveTip(ev); });
  function moveTip(ev) {
    const r=container.getBoundingClientRect();
    tip.style.left=(ev.clientX-r.left+13)+'px';
    tip.style.top =(ev.clientY-r.top -8 )+'px';
  }

  // Legend
  const lb=document.getElementById('leg-inner');
  lb.innerHTML=[...schemas.values()].slice(0,10)
    .map(({label,color})=>`<div class="lr"><div class="ld" style="background:${color}"></div>${e$(label)}</div>`)
    .join('');
  document.getElementById('legend').style.display='block';

  camFit();
  setSt(`${nodes.length} nodes · ${G.size} edges`);
}

// ═══════════════════════════════════════════════════════════════════════════
//  DETAIL PANEL
// ═══════════════════════════════════════════════════════════════════════════
function showDetail(nodeId, nRows) {
  const rows = nRows.get(nodeId)||[];
  const [tbl,col] = nodeId.split('::');
  const body = document.getElementById('rpbody');

  // Identity card
  let h = dCard('Node',[['Table',tbl],['Column',col]]);

  if (!rows.length) {
    h += `<div class="rp-hint" style="padding:12px;text-align:left;"><span style="font-size:20px">📌</span>Base source — no further lineage records.</div>`;
    body.innerHTML=h; return;
  }

  // Deduplicate
  const seen=new Set();
  const deduped=rows.filter(r=>{
    const k=`${r.sql_process}|${r.source_table}|${r.source_column}|${r.target_column}`;
    if(seen.has(k)) return false; seen.add(k); return true;
  });

  deduped.forEach((r,i)=>{
    h+=`<div class="dc"><div class="dch">${badge(r.sql_process)} Record ${i+1} / ${deduped.length}</div>`;
    h+=dr('Target Table',     r.target_table);
    h+=dr('Target Schema',    r.target_schema);
    h+=dr('Sub-Target Table', r.sub_target_table);
    h+=dr('Sub-Target Schema',r.sub_target_schema);
    h+=dr('Target Column',    r.target_column);
    h+=dr('Source Table',     r.source_table);
    h+=dr('Source Schema',    r.source_schema);
    h+=dr('Source Column',    r.source_column);
    h+=dr('Process',          r.process);
    h+=dr('SQL Process',      r.sql_process);
    if ((r.logic||'').trim())
      h+=`<div class="dr"><span class="dl">Logic</span><span class="dv mono">${e$(r.logic.trim().slice(0,400))}</span></div>`;
    h+='</div>';
  });
  body.innerHTML=h;
}

function dCard(title,fields) {
  let h=`<div class="dc"><div class="dch">${title}</div>`;
  fields.forEach(([l,v])=>{ h+=dr(l,v); });
  return h+'</div>';
}
function dr(label,val) {
  const empty=!val||!String(val).trim();
  return `<div class="dr"><span class="dl">${label}</span><span class="dv${empty?' empty':''}">${empty?'—':e$(String(val))}</span></div>`;
}
function badge(proc) {
  const m={'select':'bs','select-value':'bv','select*':'bx','join':'bj','where':'bw','having':'bh','where-subquery':'bw'};
  return `<span class="badge ${m[proc]||'bs'}">${proc||'select'}</span>`;
}

// ═══════════════════════════════════════════════════════════════════════════
//  CAMERA
// ═══════════════════════════════════════════════════════════════════════════
function camZ(f) { if(sigInst) sigInst.getCamera().animatedZoom({duration:200,factor:f}); }
function camFit() { if(sigInst) sigInst.getCamera().animatedReset({duration:400}); }

// ═══════════════════════════════════════════════════════════════════════════
//  CSV PARSER
// ═══════════════════════════════════════════════════════════════════════════
function parseCSV(text) {
  const lines=text.split(/\\r?\\n/);
  if(!lines.length) return [];
  const hdrs=csvLine(lines[0]);
  const out=[];
  for(let i=1;i<lines.length;i++){
    if(!lines[i].trim()) continue;
    const vals=csvLine(lines[i]);
    const obj={};
    hdrs.forEach((h,idx)=>{ obj[h.trim()]=(vals[idx]||'').trim(); });
    out.push(obj);
  }
  return out;
}
function csvLine(line) {
  const r=[]; let cur='',inQ=false;
  for(let i=0;i<line.length;i++){
    const c=line[i];
    if(c==='"'){ if(inQ&&line[i+1]==='"'){cur+='"';i++;} else inQ=!inQ; }
    else if(c===','&&!inQ){ r.push(cur);cur=''; }
    else cur+=c;
  }
  r.push(cur); return r;
}

// ═══════════════════════════════════════════════════════════════════════════
//  UTILS
// ═══════════════════════════════════════════════════════════════════════════
function e$(s){ return String(s||'').replace(/&/g,'&amp;').replace(/</g,'&lt;').replace(/>/g,'&gt;').replace(/"/g,'&quot;'); }
</script>
</body>
</html>
"""

with open(HTML_OUT, 'w', encoding='utf-8') as f:
    f.write(HTML_CONTENT)
print(f'✓  HTML explorer written: {HTML_OUT}')
print(f'   1. Open {HTML_OUT} in your browser')
print(f'   2. Click "📂 Load CSV / Excel" and load the lineage file')
print(f'   3. Pick a column from the dropdown, then a table, then click Trace')


## 6. Inline Preview (optional)

Renders the explorer inside the notebook. For best experience use the standalone HTML file.

> **Note:** Click **📂 Load CSV** inside the iframe and select the `_data.csv` file written above.


In [ ]:
ipy_display(IFrame(HTML_OUT, width='100%', height='750px'))


## 7. Usage Guide

| Step | Action |
|------|--------|
| 1 | Run cells 1–5 to load data and write the HTML file |
| 2 | Open `ikg_lineage_explorer.html` in a browser |
| 3 | Click **📂 Load CSV** → select `ikg_lineage_explorer_data.csv` |
| 4 | Type any column name (e.g. `acc_mhh_n`, `household_plus`, `ikg_key`) |
| 5 | Click **▶ Trace** |
| 6 | Click any node in the graph to see full details in the right panel |

### Traversal logic

Starting from every matching `sub_target_table` for the entered column, the graph
follows `source_table → sub_target_table` edges recursively until it reaches
base tables with no further lineage records.  
When `source_table` is blank, `source_column` is looked up as a `target_column`
in the next hop.

### Node colours by schema

| Schema variable | Resolved schema | Colour |
|---|---|---|
| `IKG_SCHEMA` | `core_ikg` / `sandbox_prj_smart_insights` | 🔴 Red |
| `EDW_VIEW_INPUT_SCHEMA` | `core_wma_shared` | 🟢 Teal |
| `EDW_INPUT_SCHEMA` | `core_wma_shared` | 🔵 Blue |
| `MODEL_SCHEMA` | `core_model` | 🟠 Amber |
| `NLG_SCHEMA` | `core_nlg` | 🟣 Purple |

### Column detail panel

Clicking a node shows the following fields from the lineage table:
`process`, `target_table`, `target_schema`, `sub_target_table`, `sub_target_schema`,
`source_table`, `source_schema`, `target_column`, `source_column`, `logic`, `sql_process`.
